# E-GraphSAGE (Lo et al., 2022) on UNSW-NB15, with and without the graph

**Question.** The E-GraphSAGE paper reports results on four IoT datasets and, in the text we read, does not run a control without the graph or check that training and test graphs have a similar density. Here we apply the same model design to UNSW-NB15 and add those controls.

**Model, following the paper.** Nodes are (IP, port) endpoints, edges are flows. The node feature is a constant. Two E-GraphSAGE layers with 128 hidden units, mean aggregation over the full neighbourhood (no sampling), ReLU, dropout 0.2 between the layers, a linear classifier on the concatenated source and destination embeddings, cross-entropy loss, Adam. The whole graph is processed at once (full batch). In each layer the message from a neighbour is a linear function of the neighbour embedding and the connecting edge's features, averaged over neighbours; the node update is a linear layer on [own embedding, aggregated message].

**Differences from the paper (all forced by our setting).** (1) Epochs are capped and early stopping on a validation split is used (the paper trains for a fixed, long schedule). (2) The constant node feature has size 1 instead of the edge-feature size (it is a constant either way). (3) Our train and test sets are the project's own split, and the test graph is built from test flows only.

**Controls.**

| Condition | What it shows |
|---|---|
| **no graph** | the same network where every flow is its own isolated pair of nodes. Embeddings use only the flow's own features |
| **real graph, full train graph** | what the paper's recipe gives when the whole training set is one graph (denser than the test graph) |
| **real graph, matched density** | train graph split into test-sized chunks, so train and test neighbourhoods have the same density |
| **shuffled edges, matched density** | each flow gets another flow's endpoints: same degrees, no real connections |

Each condition is run with the paper's plain cross-entropy loss and with square-root balanced class weights (the setting used in our earlier GNN notebooks). Reported metrics: macro F1 (our main metric), weighted F1 (the metric the GNN papers report), accuracy and attack-vs-normal F1. Test predictions are saved for later bootstrap analysis. Test data is never used for training or early stopping.

In [ ]:
import copy
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.checkpoint import checkpoint

DEBUG = False   # True = quick smoke test on a small random sample
SEEDS = [42] if DEBUG else [42, 43, 44]
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 128
DROPOUT = 0.2
LR = 3e-3
MAX_EPOCHS = 40 if DEBUG else 500
EVAL_EVERY = 10
PATIENCE_EVALS = 8        # stop after this many evaluations without improvement
print("device:", DEVICE, "| debug:", DEBUG, "| seeds:", SEEDS)

## 1. Load data and build graphs

The graph is stored as two sparse operators, so that a full-batch layer is a sparse matrix product:
- `A` averages the embeddings of the neighbouring nodes (every flow is a neighbour link for both of its endpoints),
- `Me` holds, for every node, the average of the features of its flows (this does not change between layers, so it is computed once).

Averaging `W [neighbour embedding, edge features]` over neighbours equals `W_h * mean(neighbour embeddings) + W_e * mean(edge features)` because the message is linear, so this is the same computation as in the paper, in a cheaper form.

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
if DEBUG:
    train_df, test_df = train_df.sample(150_000, random_state=0), test_df.sample(30_000, random_state=0)
train_graph = pd.read_csv(f"{DATASET_PATH}/train_graph.csv", index_col="row_id").loc[train_df.index]
test_graph = pd.read_csv(f"{DATASET_PATH}/test_graph.csv", index_col="row_id").loc[test_df.index]

class_names = list(joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_)
n_classes = len(class_names)
normal = class_names.index("Normal")
feature_cols = [c for c in train_df.columns if c != "label"]
N_CHUNKS = max(1, round(len(train_df) / len(test_df)))
print("train edges", len(train_df), "| test edges", len(test_df), "| features", len(feature_cols), "| chunks", N_CHUNKS)


def build_graph(graph_df, feat_df, chunk=None):
    n = len(graph_df)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ip_code = ip_code.astype("int64")
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    key = ip_code * 70_000 + ports
    if chunk is not None:                                   # separate node set per chunk
        key = key + np.concatenate([chunk, chunk]).astype("int64") * 10_000_000
    node_id, node_keys = pd.factorize(key)                  # one node per distinct (chunk, IP, port)
    n_nodes = len(node_keys)
    src, dst = torch.from_numpy(node_id[:n]).to(DEVICE), torch.from_numpy(node_id[n:]).to(DEVICE)

    nodes = torch.cat([src, dst])                           # every flow links both of its endpoints
    other = torch.cat([dst, src])
    flow = torch.cat([torch.arange(n, device=DEVICE), torch.arange(n, device=DEVICE)])
    inv_deg = 1.0 / torch.bincount(nodes, minlength=n_nodes).float()
    vals = inv_deg[nodes]
    x = torch.from_numpy(feat_df[feature_cols].to_numpy("float32")).to(DEVICE)
    A = torch.sparse_coo_tensor(torch.stack([nodes, other]), vals, (n_nodes, n_nodes)).coalesce()
    inc = torch.sparse_coo_tensor(torch.stack([nodes, flow]), vals, (n_nodes, n))
    Me = torch.sparse.mm(inc, x)                            # mean flow features per node
    return {"n_nodes": n_nodes, "src": src, "dst": dst, "A": A, "Me": Me, "x": x,
            "y": torch.from_numpy(feat_df["label"].to_numpy("int64")).to(DEVICE), "inv_deg": inv_deg}


def shuffle_endpoints(graph_df, seed):
    """Give every flow another flow's endpoints (the four columns move together)."""
    perm = np.random.default_rng(seed).permutation(len(graph_df))
    out = graph_df.copy()
    for col in ["srcip", "dstip", "sport", "dsport"]:
        out[col] = graph_df[col].to_numpy()[perm]
    return out


def chunks_for(seed):
    return np.random.default_rng(seed).integers(0, N_CHUNKS, len(train_df))


G_train_full = build_graph(train_graph, train_df)
G_test = build_graph(test_graph, test_df)
G_train_chunk = build_graph(train_graph, train_df, chunk=chunks_for(42))

## 2. Graph density

Share of nodes with a single flow, and mean flows per node. The matched chunks should look like the test graph; the full train graph is denser.

In [ ]:
density = {}
for name, G in [("train, full graph", G_train_full), ("train, matched chunks", G_train_chunk), ("test graph", G_test)]:
    deg = (1.0 / G["inv_deg"]).cpu().numpy()
    density[name] = {"nodes": G["n_nodes"], "edges": len(G["src"]), "degree-1 nodes": f"{(deg == 1).mean():.1%}",
                     "mean degree": round(float(deg.mean()), 2)}
print(pd.DataFrame(density).T.to_string())
del G_train_chunk

## 3. Model

Layer k: `message = W_h(mean of neighbour embeddings) + W_e(mean of edge features)`, then `embedding = ReLU(W_apply([own embedding, message]))`. After two layers the classifier is one linear layer on [source embedding, destination embedding].

The "no graph" condition is the same network on a graph where every flow is isolated (two private nodes, each with the other as its only neighbour). That case has a closed form per flow, so it is computed directly on the flow features.

In [ ]:
class EGraphSAGE(nn.Module):
    def __init__(self, n_feat, hidden, n_classes):
        super().__init__()
        self.msg_h = nn.ModuleList([nn.Linear(1, hidden, bias=False), nn.Linear(hidden, hidden, bias=False)])
        self.msg_e = nn.ModuleList([nn.Linear(n_feat, hidden), nn.Linear(n_feat, hidden)])
        self.update = nn.ModuleList([nn.Linear(1 + hidden, hidden), nn.Linear(2 * hidden, hidden)])
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(2 * hidden, n_classes)

    def layer(self, k, H, A, Me):
        message = self.msg_h[k](torch.sparse.mm(A, H)) + self.msg_e[k](Me)
        return torch.relu(self.update[k](torch.cat([H, message], dim=1)))

    def forward(self, G, idx):
        H = torch.ones(G["n_nodes"], 1, device=DEVICE)
        for k in range(2):
            H = checkpoint(self.layer, k, H, G["A"], G["Me"], use_reentrant=False)   # saves memory on large graphs
            if k == 0:
                H = self.drop(H)
        return self.out(torch.cat([H[G["src"][idx]], H[G["dst"][idx]]], dim=1))

    def isolated(self, x):
        """Same network when every flow is alone: neighbour = the other endpoint of the same flow."""
        H = torch.ones(len(x), 1, device=x.device)
        H = torch.relu(self.update[0](torch.cat([H, self.msg_h[0](H) + self.msg_e[0](x)], dim=1)))
        H = self.drop(H)
        H = torch.relu(self.update[1](torch.cat([H, self.msg_h[1](H) + self.msg_e[1](x)], dim=1)))
        return self.out(torch.cat([H, H], dim=1))


def logits_of(model, G, idx, use_graph):
    if use_graph:
        return model(G, idx)
    return checkpoint(model.isolated, G["x"][idx], use_reentrant=False)

## 4. One run = one condition and one seed

15% of the train edges are held out for early stopping on validation macro F1 (their labels are hidden from the loss; their features sit in the graph). The test graph is used once, at the end.

In [ ]:
def scores(y_true, pred):
    per_class = f1_score(y_true, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    return {"macro F1": per_class.mean(), "weighted F1": f1_score(y_true, pred, average="weighted", zero_division=0),
            "accuracy": accuracy_score(y_true, pred),
            "attack-vs-normal F1": f1_score(y_true == normal, pred == normal, pos_label=False, zero_division=0)}, per_class


@torch.no_grad()
def predict(model, G, idx, use_graph):
    model.eval()
    return logits_of(model, G, idx, use_graph).argmax(1).cpu().numpy()


def run(name, seed, graph, weighted, use_graph=True, shuffle=False):
    t0 = time.time()
    torch.manual_seed(seed)
    np.random.seed(seed)
    if graph == "chunk":
        G_tr = build_graph(shuffle_endpoints(train_graph, seed) if shuffle else train_graph, train_df, chunk=chunks_for(seed))
    else:
        G_tr = G_train_full

    y_all = G_tr["y"].cpu().numpy()
    tr_np, va_np = train_test_split(np.arange(len(y_all)), test_size=0.15, stratify=y_all, random_state=seed)
    tr_idx, va_idx = torch.from_numpy(tr_np).to(DEVICE), torch.from_numpy(va_np).to(DEVICE)
    weights = None
    if weighted:
        weights = torch.tensor(np.sqrt(compute_class_weight("balanced", classes=np.arange(n_classes), y=y_all[tr_np])),
                               dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    model = EGraphSAGE(len(feature_cols), HIDDEN, n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    y_val = y_all[va_np]

    best, bad, trace = {"f1": -1.0, "state": None, "epoch": 0}, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        opt.zero_grad()
        loss = loss_fn(logits_of(model, G_tr, tr_idx, use_graph), G_tr["y"][tr_idx])
        loss.backward()
        opt.step()
        if epoch % EVAL_EVERY == 0:
            val_f1 = f1_score(y_val, predict(model, G_tr, va_idx, use_graph), average="macro")
            trace.append(round(val_f1, 3))
            if val_f1 > best["f1"]:
                best, bad = {"f1": val_f1, "state": copy.deepcopy(model.state_dict()), "epoch": epoch}, 0
            else:
                bad += 1
                if bad >= PATIENCE_EVALS:
                    break

    model.load_state_dict(best["state"])
    pred = predict(model, G_test, torch.arange(len(G_test["src"]), device=DEVICE), use_graph)
    s, per_class = scores(G_test["y"].cpu().numpy(), pred)
    print(f"{name:<52} seed {seed} | best epoch {best['epoch']:>3} | val F1 {best['f1']:.4f} | test macro {s['macro F1']:.4f} "
          f"weighted {s['weighted F1']:.4f} | {time.time() - t0:.0f}s | val trace {trace[::3]}", flush=True)
    row = {"condition": name, "seed": seed, "val macro F1": best["f1"], "best epoch": best["epoch"], **s,
           **{f"F1 {c}": f for c, f in zip(class_names, per_class)}}
    del G_tr, model, opt
    torch.cuda.empty_cache()
    return row, pred.astype("int8")


GRAPHS = {
    "no graph": dict(graph="full", use_graph=False),
    "real graph, full train graph": dict(graph="full"),
    "real graph, matched density": dict(graph="chunk"),
    "shuffled edges, matched density": dict(graph="chunk", shuffle=True),
}
LOSSES = {"plain loss": False, "sqrt class weights": True}
CONDITIONS = {f"{loss} | {g}": dict(weighted=w, **kw) for loss, w in LOSSES.items() for g, kw in GRAPHS.items()}

## 5. Run all conditions

In [ ]:
rows, preds = [], {}
for seed in SEEDS:
    for name, kwargs in CONDITIONS.items():
        row, pred = run(name, seed, **kwargs)
        rows.append(row)
        preds[f"{name}__{seed}"] = pred
    pd.DataFrame(rows).to_csv("/kaggle/working/egraphsage_runs.csv", index=False)
results = pd.DataFrame(rows)
pd.DataFrame(preds, index=test_df.index).to_parquet("/kaggle/working/egraphsage_test_pred.parquet")

## 6. Results

In [ ]:
cols = ["macro F1", "weighted F1", "accuracy", "attack-vs-normal F1"]
summary = results.groupby("condition", sort=False)[cols].agg(["mean", "std"])
print(summary.round(4).to_string())
summary.to_csv("/kaggle/working/egraphsage_summary.csv")

order = list(CONDITIONS)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, col in zip(axes, ["macro F1", "weighted F1"]):
    sns.barplot(data=results, y="condition", x=col, order=order, errorbar="sd", capsize=0.15, color="steelblue", alpha=0.8, ax=ax)
    sns.stripplot(data=results, y="condition", x=col, order=order, color="black", size=5, ax=ax)
    ax.set_title(f"test {col} (mean and std over seeds)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
plt.show()

pivot = {c: results.pivot(index="seed", columns="condition", values=c) for c in ["macro F1", "weighted F1"]}
for loss in LOSSES:
    for a, b in [("real graph, matched density", "no graph"), ("real graph, matched density", "shuffled edges, matched density"),
                 ("real graph, matched density", "real graph, full train graph"), ("real graph, full train graph", "no graph")]:
        for c in ["macro F1", "weighted F1"]:
            d = pivot[c][f"{loss} | {a}"] - pivot[c][f"{loss} | {b}"]
            print(f"[{loss}] {a} minus {b} | {c}: per seed {d.round(3).tolist()} | mean {d.mean():+.4f}")